# 01 — Baseline Model

This notebook runs the Wolf-Sheep ecosystem model with default parameters and explores the resulting population dynamics.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

from simulation.runner import SimulationRunner, RunConfig
from model.config import EcosystemConfig
from model.agents import Wolf, Sheep, GrassPatch

runner = SimulationRunner()
print('Setup complete.')

## 2. Run Baseline (300 steps)

In [ ]:
cfg = EcosystemConfig()  # all defaults
rc = RunConfig(ecosystem_config=cfg, n_steps=300, seed=42, stop_on_extinction=False)
result = runner.run(rc)

print(f'Steps completed : {result.steps_completed}')
print(f'Extinction events: {result.extinction_events or "None"}')
print()
print(result.data.describe().round(1))

In [ ]:
df = result.data

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Wolves'], name='Wolves', line=dict(color='red')))
fig.add_trace(go.Scatter(x=df.index, y=df['Sheep'],  name='Sheep',  line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=df.index, y=df['Grass_Coverage_Pct'], name='Grass %', line=dict(color='green', dash='dash')))
fig.update_layout(title='Population Dynamics — Baseline', xaxis_title='Step', yaxis_title='Count / %', hovermode='x unified')
fig.show()

In [ ]:
# Phase plane: wolves vs sheep, coloured by step
fig2 = px.scatter(
    df, x='Wolves', y='Sheep', color=df.index,
    color_continuous_scale='viridis',
    title='Phase Plane — Wolves vs Sheep (colour = step)',
    labels={'color': 'Step'},
)
fig2.show()

### What does the cyclic orbit mean ecologically?

The phase plane trace forms a closed (or near-closed) orbit — the classic **Lotka-Volterra limit cycle**.

- When sheep are abundant, wolves have plenty to eat and their numbers rise.
- Increased wolf predation drives sheep numbers down.
- With fewer sheep, wolves starve and their numbers fall.
- Reduced predation pressure allows sheep to recover — and the cycle repeats.

This predator-prey oscillation is well-documented in real ecosystems: the 90-year Canadian lynx–hare dataset (MacLulich 1937) is the canonical example. The orbit's shape encodes system resilience — a tighter spiral suggests the system damps toward equilibrium; a wider orbit suggests persistent oscillation.

## 3. Grid Visualisation

In [ ]:
model = result.model
width, height = model.config.width, model.config.height

grid = np.zeros((height, width))
for agent in model.agents.select(agent_type=GrassPatch):
    x, y = agent.pos
    grid[y, x] = 1 if agent.fully_grown else 0

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(grid, cmap='Greens', origin='lower', vmin=0, vmax=1)

# Overlay agents
for a in model.agents.select(agent_type=Sheep):
    ax.plot(*a.pos, 'bo', markersize=4, alpha=0.7)
for a in model.agents.select(agent_type=Wolf):
    ax.plot(*a.pos, 'r^', markersize=6, alpha=0.9)

ax.set_title(f'Grid State — Step {model.steps}  |  Blue=Sheep  Red=Wolves  Green=Grass')
ax.set_xlabel('X'); ax.set_ylabel('Y')
plt.tight_layout()
plt.show()

### Spatial clustering

Agent-based models reveal spatial structure that aggregate ODEs cannot capture.

- **Sheep cluster** where grass is available, forming foraging patches.
- **Wolves cluster** at the edges of sheep herds, where prey is accessible.
- **Grass depletes locally** around high sheep density, creating a moving mosaic of bare ground and recovering patches.

This local depletion / recovery dynamic is ecologically significant: it slows the global oscillation compared to the mean-field ODE prediction, because predators must search spatially for prey rather than encountering them uniformly.

## 4. Extinction Experiment

In [ ]:
# Near-zero wolf food gain → trophic collapse
collapse_cfg = EcosystemConfig(wolf_gain_from_food=2, initial_wolf_energy=4)
collapse_rc = RunConfig(ecosystem_config=collapse_cfg, n_steps=300, seed=42, stop_on_extinction=True)
collapse_result = runner.run(collapse_rc)

print(f'Terminated early: {collapse_result.terminated_early}')
print(f'Extinct species : {collapse_result.extinction_events}')
print(f'Steps completed : {collapse_result.steps_completed}')

cdf = collapse_result.data
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=cdf.index, y=cdf['Wolves'], name='Wolves', line=dict(color='red')))
fig3.add_trace(go.Scatter(x=cdf.index, y=cdf['Sheep'],  name='Sheep',  line=dict(color='steelblue')))
fig3.update_layout(title='Population Collapse (wolf_gain_from_food=2)', xaxis_title='Step', yaxis_title='Count')
fig3.show()

### Real-world trophic cascades

When a top predator is removed (or energy-starved as in this experiment), the ecosystem does not simply lose one species — it restructures:

- **Yellowstone wolves** (reintroduced 1995): wolf removal in the early 20th century led to elk overgrazing of riverbanks, causing erosion and loss of riparian habitat. Reintroduction reversed this — a textbook *trophic cascade*.
- **Sea otters / urchins / kelp** (Pacific coast): otter hunting → urchin explosion → kelp forest collapse.
- **UK Sparrowhawk**: DDT-driven decline 1950s–70s caused garden bird populations to shift; Sparrowhawk recovery since 1986 ban has re-exerted predation pressure.

This model's wolf-starvation experiment mimics the *bottom-up* side: low energy gain → wolf extinction → unchecked sheep growth → grass exhaustion. Adjust `sheep_reproduce` and `grass_regrowth_time` to explore whether prey can overshoot without predators.